# Phase 4 — Model Evaluation & Explainability

**Project:** Hospital Operations & Revenue Risk Intelligence Platform  
**Goal:** Ensure predictions are reliable, interpretable, and safe for hospital operations and finance teams.

---

## Notebook Structure
1. Load Models & Data
2. Model A — Technical Evaluation
3. Model A — Business Metrics & Fairness
4. Model B — Technical Evaluation
5. Model B — Business Metrics & Fairness
6. Explainability Summary
7. Model Card

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, f1_score, recall_score, accuracy_score
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13})

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('../data_outputs/model_table.csv', parse_dates=['visit_date'])
with open('../data_outputs/feature_schema.json') as f:
    schema = json.load(f)

# ── Time-based split ──────────────────────────────────────────────────────────
df_sorted  = df.sort_values('visit_date').reset_index(drop=True)
split_idx  = int(len(df_sorted) * 0.80)
train_df   = df_sorted.iloc[:split_idx]
test_df    = df_sorted.iloc[split_idx:]

# ── Load best models ──────────────────────────────────────────────────────────
model_a = joblib.load('../data_outputs/model_a_risk.joblib')
model_b = joblib.load('../data_outputs/model_b_claim.joblib')

print(f'Models loaded: Model A ({schema["best_model_a"]}), Model B ({schema["best_model_b"]})')
print(f'Test set: {len(test_df):,} rows')

## 2. Model A — Technical Evaluation

> **Business context:** High-Risk visit misclassification has direct patient safety implications. Recall for the High class is the primary safety metric.

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
FEATURES_A = schema['model_a_risk_features']
TARGET_A   = schema['target_model_a']

test_a   = test_df.dropna(subset=[TARGET_A])
X_test_a = test_a[FEATURES_A]
y_test_a = test_a[TARGET_A]

# Train predictions (for overfit check)
train_a   = train_df.dropna(subset=[TARGET_A])
X_train_a = train_a[FEATURES_A]
y_train_a = train_a[TARGET_A]

y_pred_train_a = model_a.predict(X_train_a)
y_pred_test_a  = model_a.predict(X_test_a)

print('=== MODEL A: Train Performance ===')
print(f'Accuracy: {accuracy_score(y_train_a, y_pred_train_a):.4f}')
print(classification_report(y_train_a, y_pred_train_a))

print('\n=== MODEL A: Test Performance ===')
print(f'Accuracy: {accuracy_score(y_test_a, y_pred_test_a):.4f}')
print(classification_report(y_test_a, y_pred_test_a))

In [ ]:
# ── Confusion matrix with % ───────────────────────────────────────────────────
labels_a = sorted(y_test_a.unique())
cm_a = confusion_matrix(y_test_a, y_pred_test_a, labels=labels_a)
cm_a_pct = cm_a.astype(float) / cm_a.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm_a, display_labels=labels_a).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Model A — Confusion Matrix (Counts)')

sns.heatmap(cm_a_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=labels_a, yticklabels=labels_a, ax=axes[1])
axes[1].set_title('Model A — Confusion Matrix (Row %)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../data_outputs/eval_modelA_cm.png', dpi=110); plt.show()

## 3. Model A — Business Metrics & Fairness

In [ ]:
# ── Business metric: High-Risk Recall ─────────────────────────────────────────
prec, rec, f1, sup = precision_recall_fscore_support(y_test_a, y_pred_test_a, labels=labels_a)

business_df_a = pd.DataFrame({
    'Class':     labels_a,
    'Precision': prec.round(3),
    'Recall':    rec.round(3),
    'F1-Score':  f1.round(3),
    'Support':   sup
})

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(labels_a))
w = 0.25
ax.bar(x - w, prec, w, label='Precision', color='steelblue')
ax.bar(x,     rec,  w, label='Recall',    color='seagreen')
ax.bar(x + w, f1,   w, label='F1-Score',  color='tomato')
ax.set_xticks(x); ax.set_xticklabels(labels_a)
ax.set_ylim(0, 1.15); ax.set_title('Model A — Precision / Recall / F1 by Class')
ax.axhline(0.7, linestyle='--', color='grey', label='0.7 threshold')
ax.legend(); plt.tight_layout()
plt.savefig('../data_outputs/eval_modelA_prf.png', dpi=110); plt.show()

display(business_df_a)
high_idx = list(labels_a).index('High') if 'High' in labels_a else -1
if high_idx >= 0:
    print(f'\n🎯 KEY BUSINESS METRIC — High-Risk Recall: {rec[high_idx]:.4f}')
    print(f'   Target: ≥ 0.70 | Status: {"✅ PASS" if rec[high_idx] >= 0.70 else "⚠️ BELOW TARGET"}')

In [ ]:
# ── Fairness: Performance by gender ──────────────────────────────────────────
test_a_full = test_a.copy()
test_a_full['predicted'] = y_pred_test_a

def segment_recall(df_seg, true_col, pred_col, seg_col, target_class):
    results = []
    for seg_val in df_seg[seg_col].unique():
        subset = df_seg[df_seg[seg_col] == seg_val]
        if target_class in subset[true_col].values:
            r = recall_score(subset[true_col], subset[pred_col],
                             labels=[target_class], average='macro', zero_division=0)
            results.append({seg_col: seg_val, 'recall_high_risk': round(r, 3), 'n': len(subset)})
    return pd.DataFrame(results).sort_values('recall_high_risk', ascending=False)

# By gender
fair_gender = segment_recall(test_a_full, TARGET_A, 'predicted', 'gender', 'High')
# By city
fair_city   = segment_recall(test_a_full, TARGET_A, 'predicted', 'city',   'High')
# By insurance_provider
fair_ins    = segment_recall(test_a_full, TARGET_A, 'predicted', 'insurance_provider', 'High')

print('Fairness — High-Risk Recall by Gender:')
display(fair_gender)
print('\nFairness — High-Risk Recall by City (Top 5):')
display(fair_city.head())
print('\nFairness — High-Risk Recall by Insurer (Top 5):')
display(fair_ins.head())

In [ ]:
# ── Fairness visualization ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=fair_city.head(8), y='city', x='recall_high_risk', ax=axes[0], palette='Blues_d')
axes[0].axvline(0.7, color='red', linestyle='--', label='0.7 target')
axes[0].set_title('High-Risk Recall by City'); axes[0].legend()

sns.barplot(data=fair_ins, y='insurance_provider', x='recall_high_risk', ax=axes[1], palette='Greens_d')
axes[1].axvline(0.7, color='red', linestyle='--', label='0.7 target')
axes[1].set_title('High-Risk Recall by Insurer'); axes[1].legend()

plt.tight_layout()
plt.savefig('../data_outputs/eval_modelA_fairness.png', dpi=110); plt.show()
print('\n📊 Fairness Insight: Any segment below 0.70 represents an equity gap requiring model retraining or rule-based overrides.')

## 4. Model B — Technical Evaluation

In [ ]:
FEATURES_B = schema['model_b_claim_features']
TARGET_B   = schema['target_model_b']

test_b    = test_df.dropna(subset=[TARGET_B])
train_b   = train_df.dropna(subset=[TARGET_B])
X_test_b  = test_b[FEATURES_B]
y_test_b  = test_b[TARGET_B]
X_train_b = train_b[FEATURES_B]
y_train_b = train_b[TARGET_B]

y_pred_train_b = model_b.predict(X_train_b)
y_pred_test_b  = model_b.predict(X_test_b)

print('=== MODEL B: Train Performance ===')
print(f'Accuracy: {accuracy_score(y_train_b, y_pred_train_b):.4f}')
print(classification_report(y_train_b, y_pred_train_b))

print('\n=== MODEL B: Test Performance ===')
print(f'Accuracy: {accuracy_score(y_test_b, y_pred_test_b):.4f}')
print(classification_report(y_test_b, y_pred_test_b))

In [ ]:
# ── Model B confusion matrix ──────────────────────────────────────────────────
labels_b = sorted(y_test_b.unique())
cm_b = confusion_matrix(y_test_b, y_pred_test_b, labels=labels_b)
cm_b_pct = cm_b.astype(float) / cm_b.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cm_b, display_labels=labels_b).plot(ax=axes[0], colorbar=False, cmap='Oranges')
axes[0].set_title('Model B — Confusion Matrix (Counts)')

sns.heatmap(cm_b_pct, annot=True, fmt='.1f', cmap='Oranges',
            xticklabels=labels_b, yticklabels=labels_b, ax=axes[1])
axes[1].set_title('Model B — Confusion Matrix (Row %)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../data_outputs/eval_modelB_cm.png', dpi=110); plt.show()

## 5. Model B — Business Metrics & Fairness

In [ ]:
# ── Business metric: Rejected Recall ─────────────────────────────────────────
prec_b, rec_b, f1_b, sup_b = precision_recall_fscore_support(
    y_test_b, y_pred_test_b, labels=labels_b)

business_df_b = pd.DataFrame({
    'Class':     labels_b,
    'Precision': prec_b.round(3),
    'Recall':    rec_b.round(3),
    'F1-Score':  f1_b.round(3),
    'Support':   sup_b
})

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(labels_b))
w = 0.25
ax.bar(x - w, prec_b, w, label='Precision', color='steelblue')
ax.bar(x,     rec_b,  w, label='Recall',    color='seagreen')
ax.bar(x + w, f1_b,   w, label='F1-Score',  color='tomato')
ax.set_xticks(x); ax.set_xticklabels(labels_b)
ax.set_ylim(0, 1.15); ax.set_title('Model B — Precision / Recall / F1 by Class')
ax.axhline(0.7, linestyle='--', color='grey', label='0.7 threshold')
ax.legend(); plt.tight_layout()
plt.savefig('../data_outputs/eval_modelB_prf.png', dpi=110); plt.show()

display(business_df_b)
rej_idx = list(labels_b).index('Rejected') if 'Rejected' in labels_b else -1
if rej_idx >= 0:
    print(f'\n🎯 KEY BUSINESS METRIC — Rejected Recall: {rec_b[rej_idx]:.4f}')
    print(f'   Target: ≥ 0.70 | Status: {"✅ PASS" if rec_b[rej_idx] >= 0.70 else "⚠️ BELOW TARGET"}')

In [ ]:
# ── Fairness by insurance provider (Model B) ──────────────────────────────────
test_b_full = test_b.copy()
test_b_full['predicted'] = y_pred_test_b

fair_ins_b = segment_recall(test_b_full, TARGET_B, 'predicted', 'insurance_provider', 'Rejected')
fair_city_b = segment_recall(test_b_full, TARGET_B, 'predicted', 'city', 'Rejected')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=fair_ins_b, y='insurance_provider', x='recall_high_risk', ax=axes[0], palette='Reds_d')
axes[0].axvline(0.7, color='blue', linestyle='--'); axes[0].set_title('Rejected Recall by Insurer')

sns.barplot(data=fair_city_b.head(8), y='city', x='recall_high_risk', ax=axes[1], palette='Purples_d')
axes[1].axvline(0.7, color='blue', linestyle='--'); axes[1].set_title('Rejected Recall by City')

plt.tight_layout()
plt.savefig('../data_outputs/eval_modelB_fairness.png', dpi=110); plt.show()

## 6. Explainability Summary

In [ ]:
# ── Feature importance for both models (if tree-based) ────────────────────────
def plot_feature_importance(model, features, title, filename, cat_cols, num_cols, bin_cols, top_n=15):
    try:
        ohe = model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        f_num = [c for c in num_cols if c in features]
        f_cat = [c for c in cat_cols if c in features]
        f_bin = [c for c in bin_cols if c in features]
        cat_names = list(ohe.get_feature_names_out(f_cat))
        all_names = f_num + cat_names + f_bin
        imps      = model.named_steps['classifier'].feature_importances_
        fi = pd.DataFrame({'feature': all_names, 'importance': imps})
        fi = fi.sort_values('importance', ascending=False).head(top_n)
        
        fig, ax = plt.subplots(figsize=(9, 6))
        sns.barplot(data=fi, y='feature', x='importance', ax=ax, palette='viridis')
        ax.set_title(title); plt.tight_layout()
        plt.savefig(f'../data_outputs/{filename}', dpi=110); plt.show()
        return fi
    except AttributeError:
        print(f'  Model does not have feature_importances_ — likely Logistic Regression')
        return None

cat_cols = schema['categorical_cols']
num_cols = schema['numeric_cols']
bin_cols = schema['binary_cols']

print('=== Model A Feature Importances ===')
fi_a = plot_feature_importance(model_a, FEATURES_A,
    'Model A — Top 15 Feature Importances (Risk Classifier)',
    'eval_fi_modelA.png', cat_cols, num_cols, bin_cols)

print('\n=== Model B Feature Importances ===')
fi_b = plot_feature_importance(model_b, FEATURES_B,
    'Model B — Top 15 Feature Importances (Claim Classifier)',
    'eval_fi_modelB.png', cat_cols, num_cols, bin_cols)

## 7. Model Card

In [ ]:
# ── Generate consolidated model card ─────────────────────────────────────────
model_card = {
    'model_card_version': '1.0',
    'generated_at': pd.Timestamp.now().isoformat(),

    'model_a': {
        'name':          'Visit Risk Classifier',
        'algorithm':     schema.get('best_model_a', 'Random Forest'),
        'target':        'risk_score',
        'classes':       ['High', 'Low', 'Medium'],
        'train_size':    len(train_df),
        'test_size':     len(test_df),
        'split_method':  'Time-based (80/20)',
        'imbalance_strategy': 'class_weight=balanced',
        'primary_business_metric': 'Recall for High-Risk class',
        'input_features': FEATURES_A,
        'limitations': [
            'Does not incorporate real-time clinical vitals',
            'Performance may degrade for rare departments with low training data',
            'Requires retraining every quarter with fresh visit data'
        ],
        'assumptions': [
            'risk_score labels are accurate ground truth',
            'Patient demographics remain stable over deployment window'
        ]
    },

    'model_b': {
        'name':          'Insurance Claim Outcome Classifier',
        'algorithm':     schema.get('best_model_b', 'Random Forest'),
        'target':        'claim_status',
        'classes':       ['Paid', 'Pending', 'Rejected'],
        'train_size':    len(train_df),
        'test_size':     len(test_df),
        'split_method':  'Time-based (80/20)',
        'imbalance_strategy': 'Random oversampling + class_weight=balanced',
        'primary_business_metric': 'Recall for Rejected class',
        'input_features': FEATURES_B,
        'data_leakage_note': 'approved_amount excluded — available only post-settlement',
        'limitations': [
            'Does not encode insurer-specific contract rules',
            'Pending class performance is inherently uncertain',
            'New insurers not seen in training will use OHE unknown handling'
        ],
        'assumptions': [
            'Billing patterns remain consistent across quarters',
            'claim_status is final (no re-adjudication after labeling)'
        ]
    },

    'retraining_strategy': {
        'trigger':    'Quarterly or when prediction drift > 5% on monitoring dashboard',
        'process':    'Retrain on rolling 12-month window of visit + billing data',
        'validation': 'Shadow mode for 2 weeks before promotion to production'
    },

    'governance': {
        'model_owner':  'Hospital Analytics Team',
        'review_cycle': 'Quarterly',
        'audit_log':    'All predictions logged with timestamp, model version, input hash'
    }
}

import json
with open('../data_outputs/model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)

print('✅ Model card saved to data_outputs/model_card.json')
print(json.dumps(model_card, indent=2))

In [ ]:
# ── Evaluation summary table ──────────────────────────────────────────────────
print('\n' + '='*65)
print('FINAL EVALUATION SUMMARY')
print('='*65)
print(f'  MODEL A — Visit Risk Classifier ({schema.get("best_model_a", "RF")})')
print(f'    Test Accuracy         : {accuracy_score(y_test_a, y_pred_test_a):.4f}')
print(f'    Macro F1-Score        : {f1_score(y_test_a, y_pred_test_a, average="macro"):.4f}')
if high_idx >= 0:
    print(f'    High-Risk Recall [🎯] : {rec[high_idx]:.4f}')

print(f'\n  MODEL B — Claim Outcome Classifier ({schema.get("best_model_b", "RF")})')
print(f'    Test Accuracy         : {accuracy_score(y_test_b, y_pred_test_b):.4f}')
print(f'    Macro F1-Score        : {f1_score(y_test_b, y_pred_test_b, average="macro"):.4f}')
if rej_idx >= 0:
    print(f'    Rejected Recall [🎯]  : {rec_b[rej_idx]:.4f}')
print('='*65)
print('\n✅ Phase 4 Complete. Proceed to API deployment (Phase 5).')

---

## Phase 4 Summary

| Evaluation Dimension | Model A | Model B |
|---|---|---|
| Train vs Test gap | Checked for overfit | Checked for overfit |
| Primary business metric | High-Risk Recall | Rejected Recall |
| Fairness check | Gender, City, Insurer | City, Insurer |
| Explainability | Feature importances | Feature importances |
| Model card | ✅ | ✅ |

**Governance flags:**
- Any fairness gap below 0.70 must be reviewed before deployment
- Model card must be updated with every retraining cycle
- Predictions must be logged for quarterly drift review